In [16]:
using LowLevelFEM

In [17]:
gmsh.initialize()

structured_rect_mesh(lx=1)

In [18]:
mat = Material("surface", E=1, ν=0.3)
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f);
#Pu = Problem([mat], type=:PlaneStress);

In [19]:
supp1 = BoundaryCondition("left", ux=0)
supp2 = BoundaryCondition("bottom", uy=0)

load = LoadCondition("right", fx=1);

In [20]:
E = mat.E
ν = mat.ν

μ = mat.μ
λ = mat.λ

#D = E / ((1 + ν) * (1 - 2ν)) * [1-ν ν 0; ν 1-ν 0; 0 0 (1-2ν)/2]
D = E / (1 - ν^2) * [1 ν 0; ν 1 0; 0 0 (1-ν)/2]

3×3 Matrix{Float64}:
 1.0989   0.32967  0.0
 0.32967  1.0989   0.0
 0.0      0.0      0.384615

In [21]:
n = ScalarField(Pu, "surface", 0.0)
d = ScalarField(Pu, "surface", 1.0)
D = [d 0 n; n d n; n n 1]

3×3 Matrix{Any}:
 ScalarField([[1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;]  …  [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;], [1.0; 1.0; 1.0; 1.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54  …  135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 1, :scalar, Problem("structured_rect", :VectorField, 2, 2, Material[Material("surface", :Hooke, 1.0, 0.3, 0.576923, 0.384615, 0.833333, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f))  …   ScalarField([[0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0; 0.0;;]

In [22]:
#K = ∫(ε(Pu) ⋅ ε(Pu))
K = ∫(ε(Pu) ⋅ D ⋅ D ⋅ ε(Pu))
#K = ∫(ε(Pu) ⋅ ε(Pu) * 2μ + Div(Pu) ⋅ Div(Pu) * λ)
#K = symmetricGradientMatrix(Pu, coefficient=2μ) + gradDivMatrix(Pu, coefficient=λ)
#K = stiffnessMatrix(Pu)
K[:, :]

242×242 SparseArrays.SparseMatrixCSC{Float64, Int64} with 3764 stored entries:
⎡⡱⢎⡀⠀⠒⠀⠀⠶⠀⠀⠤⠀⠀⣉⠀⠠⡄⠀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⠂⠀⠰⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠀⠀⠁⠀⠀⠃⠀⠀⠇⠀⠀⠆⠀⠀⡆⠀⠀⡄⠀⠀⡀⠀⢀⡀⠀⠀⎥
⎢⠘⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⠁⠀⠸⢷⣄⠀⎥
⎢⢠⡄⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠀⠀⢠⠀⠙⢷⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⢀⡀⠀⢠⡄⠀⢠⡄⠀⢰⠀⠀⠸⠀⠀⠘⠀⠀⠘⠀⠀⠈⠀⠀⠀⎥
⎢⠀⠃⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⣠⡾⠃⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡄⢠⡄⠀⠀⠀⠀⠀⠀⠀⠀⠈⣻⣾⡛⠀⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⡀⠀⠀⠀⠀⠀⠀⠀⢀⣠⡾⠛⠈⠻⣦⡘⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠉⠁⠀⠀⠀⠀⠀⠀⠈⠉⠀⠀⠻⣶⡈⠻⣦⡈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠠⠤⠀⠀⠀⠀⠀⠀⠶⠆⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠻⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠉⠁⠀⠀⠀⠀⠀⠉⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠠⠄⠀⠀⠀⠀⠐⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣦⡌⠛⣤⡉⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣧⠈⠻⣦⠈⠻⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠈⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠻⣦⡀⠻⣦⡀⠻⣦⡀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠤⠄⠀⠀⠀⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡈⠛⣤⡈⠛⣤⡀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⢀⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⠈⠻⣦⠘⢻⣦⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠻⣶⣀⠻⣦⡘⠻⣦⡀⎥
⎢⠰⠀⠀⠰⢶⣆⠀⠒⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣶⡈⠻⣦⡈⠛⎥
⎣⢀⡀⠀⠀⠀⠙⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⠈⠻⣦⎦

In [23]:
f = loadVector(Pu, [load])

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [24]:
u = solveField(K, f, support=[supp1, supp2])

nodal VectorField
[0.0; 0.0; … ; 0.9000000000000068; -2.248749739293403e-15;;]

In [25]:
showDoFResults(u, name="u", visible=true, factor=1)
showDoFResults(u, :ux, name="ux")
showDoFResults(u, :uy, name="uy")

2

In [26]:
u3 = expandTo3D(u)
ex = ∂x(u3[1])
ey = ∂y(u3[2])

elementwise ScalarField
[[1.9489465402671707e-15; 1.897189156718632e-15; 1.8971891567186322e-15; 1.9489465402671707e-15;;], [2.0984422809265848e-15; 1.958259993348883e-15; 1.958259993348883e-15; 2.0984422809265848e-15;;], [2.3317351174084937e-15; 2.2730588497285334e-15; 2.2730588497285334e-15; 2.3317351174084937e-15;;], [2.3733808330345497e-15; 2.5003935309941262e-15; 2.5003935309941266e-15; 2.3733808330345497e-15;;], [2.470206299050226e-15; 2.261246163971723e-15; 2.261246163971723e-15; 2.470206299050226e-15;;], [1.7663208693885663e-15; 1.783139425660794e-15; 1.783139425660794e-15; 1.7663208693885663e-15;;], [1.059945084071604e-15; 1.0082049945997722e-15; 1.0082049945997722e-15; 1.059945084071604e-15;;], [8.780927489110275e-17; 8.299302613938932e-17; 8.299302613938932e-17; 8.780927489110275e-17;;], [-2.7749961830674827e-16; -3.258298545455491e-16; -3.258298545455491e-16; -2.7749961830674827e-16;;], [-8.630801821258474e-17; -3.833894021317856e-17; -3.833894021317856e-17; -8.630801821258

In [27]:
showDoFResults(ex, name="εx")
showDoFResults(ey, name="εy")

4

In [28]:
sx = E / (1 - ν^2) * (ex + ν * ey)
sy = E / (1 - ν^2) * (ey + ν * ex)

elementwise ScalarField
[[0.3296703296703331; 0.32967032967033305; 0.32967032967033294; 0.3296703296703331;;], [0.3296703296703333; 0.3296703296703331; 0.32967032967033316; 0.32967032967033333;;], [0.3296703296703336; 0.32967032967033355; 0.32967032967033355; 0.3296703296703336;;], [0.32967032967033366; 0.3296703296703338; 0.32967032967033394; 0.3296703296703338;;], [0.32967032967033383; 0.3296703296703337; 0.32967032967033383; 0.32967032967033405;;], [0.32967032967033333; 0.32967032967033333; 0.3296703296703339; 0.3296703296703339;;], [0.3296703296703331; 0.32967032967033305; 0.3296703296703334; 0.32967032967033344;;], [0.32967032967033244; 0.3296703296703324; 0.32967032967033294; 0.329670329670333;;], [0.32967032967033255; 0.3296703296703325; 0.32967032967033266; 0.3296703296703327;;], [0.32967032967033294; 0.329670329670333; 0.3296703296703332; 0.32967032967033316;;]  …  [0.32967032967032506; 0.32967032967032456; 0.3296703296703257; 0.3296703296703262;;], [0.3296703296703265; 0.3296

In [29]:
showDoFResults(sx, name="σx")
showDoFResults(sy, name="σy")

6

In [30]:
openPostProcessor()

In [16]:
gmsh.finalize()